In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Notebook 4 — Pipeline Orchestrator (SQL Refactored)
# MAGIC Runs Bronze, Silver, and Gold in sequence using Spark SQL.

# COMMAND ----------
# MAGIC %md ## 0. Read run mode & Setup

# COMMAND ----------

try:
    MODE = dbutils.widgets.get("mode")
except Exception:
    MODE = "full_refresh"

print(f"Run mode: {MODE}")

from datetime import datetime

CATALOG = "workspace"
SCHEMA  = "medallion_sql_pipeline"

current_user = spark.sql("SELECT current_user()").collect()[0][0]

# RAW_PATH = f"/Workspace/Users/{current_user}/raw_data"
RAW_PATH =f"/Volumes/workspace/default/course_data/raw_data"
RUN_ID   = datetime.now().strftime("%Y%m%d_%H%M%S")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE {CATALOG}.{SCHEMA}")

print(f"Catalog : {CATALOG}")
print(f"Schema  : {SCHEMA}")
print(f"Raw path: {RAW_PATH}")

# COMMAND ----------
# MAGIC %md ## 1. Metadata tables & Shared Helpers

# COMMAND ----------

# --- Metadata Tables ---
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA}.pipeline_watermarks (
    table_name      STRING,
    last_order_date DATE,
    rows_processed  BIGINT,
    run_status      STRING,
    run_mode        STRING,
    updated_at      TIMESTAMP
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA}.pipeline_run_log (
    run_id       STRING,
    run_mode     STRING,
    stage        STRING,
    rows_in      BIGINT,
    rows_out     BIGINT,
    status       STRING,
    error_msg    STRING,
    started_at   TIMESTAMP,
    finished_at  TIMESTAMP
) USING DELTA
""")

def get_watermark(table_name: str) -> str:
    rows = spark.sql(f"""
        SELECT CAST(last_order_date AS STRING) AS last_order_date
        FROM {CATALOG}.{SCHEMA}.pipeline_watermarks
        WHERE table_name = '{table_name}' AND run_status = 'SUCCESS'
        ORDER BY updated_at DESC LIMIT 1
    """).collect()
    wm = rows[0][0] if rows else "1900-01-01"
    print(f"  Watermark [{table_name}]: {wm}")
    return wm

def set_watermark(table_name: str, last_date: str, rows: int, status="SUCCESS"):
    spark.sql(f"""
        INSERT INTO {CATALOG}.{SCHEMA}.pipeline_watermarks
        VALUES ('{table_name}', DATE('{last_date}'), {rows}, '{status}', '{MODE}', CURRENT_TIMESTAMP())
    """)
    print(f"  Watermark updated [{table_name}] → {last_date} ({rows:,} rows, {status})")

def log(stage, rows_in=0, rows_out=0, status="SUCCESS", error=""):
    safe_error = (error or "")[:200].replace("'", "")
    spark.sql(f"""
        INSERT INTO {CATALOG}.{SCHEMA}.pipeline_run_log
        VALUES ('{RUN_ID}', '{MODE}', '{stage}', {rows_in}, {rows_out}, '{status}', '{safe_error}', CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP())
    """)

# --- Helpers ---
def read_raw_csv(filename):
    return (
        spark.read.option("header", "true")
        .option("inferSchema", "false")
        .option("multiLine", "true")
        .option("escape", '"')
        .csv(f"{RAW_PATH}/{filename}")
    )

def standardize_columns(df):
    def clean_name(c):
        for char in[" ", ",", ";", "{", "}", "(", ")", "\n", "\t", "=", "-"]:
            c = c.replace(char, "_")
        return c.strip()
    return df.toDF(*[clean_name(c) for c in df.columns])

# COMMAND ----------
# MAGIC %md ## 2. Bronze stage

# COMMAND ----------

def run_bronze():
    print("\n▓▓▓  STAGE 1/3 — BRONZE  ▓▓▓")
    start = datetime.now()

    def ingest_dim(filename, table_name):
        full_table_name = f"bronze_{table_name}"
        df = standardize_columns(read_raw_csv(filename))
        df.createOrReplaceTempView("temp_raw")
        
        spark.sql(f"""
            CREATE OR REPLACE TABLE {full_table_name} AS 
            SELECT *, CURRENT_TIMESTAMP() AS _ingest_time, '{filename}' AS _source_file 
            FROM temp_raw
        """)
        n = spark.table(full_table_name).count()
        print(f"  {full_table_name:45s} {n:>8,} rows")

    # Dimensions always full refresh
    ingest_dim("RAW_Customer.csv", "customer")
    ingest_dim("RAW_Product.csv", "product")
    ingest_dim("RAW_Reseller.csv", "reseller")
    ingest_dim("RAW_SalesTerritory.csv", "sales_territory")
    ingest_dim("RAW_Date.csv", "date")
    ingest_dim("RAW_SalesOrder.csv", "sales_order")

    # Fact table (Incremental)
    df_sales = standardize_columns(read_raw_csv("RAW_Sales.csv"))
    df_sales.createOrReplaceTempView("temp_raw_sales")

    wm = get_watermark("bronze_sales") if MODE == "incremental" else "1900-01-01"

    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW new_sales AS
        SELECT *, CURRENT_TIMESTAMP() AS _ingest_time, 'RAW_Sales.csv' AS _source_file
        FROM temp_raw_sales
        WHERE COALESCE(TRY_TO_DATE(Order_Date, 'yyyy-MM-dd'), TRY_TO_DATE(Order_Date, 'MM/dd/yyyy')) > DATE('{wm}')
    """)

    new_rows = spark.sql("SELECT COUNT(*) FROM new_sales").collect()[0][0]

    if new_rows == 0:
        print("  bronze_sales                                  0 new rows — skipping")
        log("bronze", rows_out=0)
        return

    if MODE == "full_refresh":
        spark.sql("CREATE OR REPLACE TABLE bronze_sales AS SELECT * FROM new_sales")
    else:
        spark.sql("INSERT INTO bronze_sales SELECT * FROM new_sales")

    max_date = spark.sql("SELECT MAX(COALESCE(TRY_TO_DATE(Order_Date, 'yyyy-MM-dd'), TRY_TO_DATE(Order_Date, 'MM/dd/yyyy'))) FROM new_sales").collect()[0][0]
    set_watermark("bronze_sales", str(max_date), new_rows)

    print(f"  bronze_sales                                  {new_rows:>8,} rows  (up to {max_date})")

    elapsed = (datetime.now() - start).seconds
    log("bronze", rows_out=new_rows)
    print(f"  ✔ Bronze done in {elapsed}s")

# COMMAND ----------
# MAGIC %md ## 3. Silver stage

# COMMAND ----------

def run_silver():
    print("\n▓▓▓  STAGE 2/3 — SILVER  ▓▓▓")
    start = datetime.now()

    spark.sql("""
        CREATE OR REPLACE TABLE silver_dim_customer AS
        SELECT 
            ROW_NUMBER() OVER(ORDER BY Customer_ID) AS CustomerKey,
            Customer_ID AS CustomerNaturalKey,
            COALESCE(NULLIF(TRIM(Customer), ''), 'Unknown') AS CustomerName,
            COALESCE(NULLIF(TRIM(City), ''), 'Unknown') AS City,
            INITCAP(TRIM(State_Province)) AS StateProvince,
            INITCAP(TRIM(Country_Region)) AS CountryRegion,
            COALESCE(NULLIF(TRIM(Postal_Code), ''), '00000') AS PostalCode
        FROM (SELECT DISTINCT Customer_ID, Customer, City, State_Province, Country_Region, Postal_Code FROM bronze_customer WHERE Customer_ID IS NOT NULL)
    """)
    print(f"  silver_dim_customer                           {spark.table('silver_dim_customer').count():>8,} rows")

    spark.sql("""
        CREATE OR REPLACE TABLE silver_dim_product AS
        SELECT 
            ROW_NUMBER() OVER(ORDER BY SKU) AS ProductKey, SKU,
            TRIM(FIRST(Product)) AS Product,
            TRIM(FIRST(Model)) AS Model,
            COALESCE(INITCAP(NULLIF(TRIM(FIRST(Category)), '')), 'Unknown') AS Category,
            INITCAP(TRIM(FIRST(Subcategory))) AS Subcategory,
            COALESCE(INITCAP(NULLIF(TRIM(FIRST(Color)), '')), 'No Color') AS Color,
            TRY_CAST(REGEXP_REPLACE(FIRST(List_Price), '[^\\d\\.\\-]', '') AS DOUBLE) AS ListPrice,
            COALESCE(TRY_CAST(REGEXP_REPLACE(FIRST(Standard_Cost), '[^\\d\\.\\-]', '') AS DOUBLE), 0.0) AS StandardCost
        FROM bronze_product WHERE SKU IS NOT NULL
        GROUP BY SKU
        HAVING ListPrice > 0
    """)
    print(f"  silver_dim_product                            {spark.table('silver_dim_product').count():>8,} rows")

    spark.sql("""
        CREATE OR REPLACE TABLE silver_dim_reseller AS
        SELECT 
            ROW_NUMBER() OVER(ORDER BY Reseller_ID) AS ResellerKey,
            Reseller_ID AS ResellerNaturalKey, TRIM(Reseller) AS Reseller,
            COALESCE(INITCAP(NULLIF(TRIM(Business_Type), '')), 'Unknown') AS BusinessType,
            COALESCE(NULLIF(TRIM(City), ''), 'Unknown') AS City,
            INITCAP(TRIM(State_Province)) AS StateProvince,
            INITCAP(TRIM(Country_Region)) AS CountryRegion,
            COALESCE(NULLIF(TRIM(Postal_Code), ''), '00000') AS PostalCode
        FROM (SELECT DISTINCT Reseller_ID, Reseller, Business_Type, City, State_Province, Country_Region, Postal_Code FROM bronze_reseller WHERE Reseller_ID IS NOT NULL)
    """)
    print(f"  silver_dim_reseller                           {spark.table('silver_dim_reseller').count():>8,} rows")

    spark.sql("""
        CREATE OR REPLACE TABLE silver_dim_territory AS
        SELECT 
            ROW_NUMBER() OVER(ORDER BY Region) AS SalesTerritoryKey,
            INITCAP(TRIM(Region)) AS Region, INITCAP(TRIM(Country)) AS Country, INITCAP(TRIM(Sales_Group)) AS SalesGroup
        FROM (SELECT DISTINCT Region, Country, Sales_Group FROM bronze_sales_territory WHERE Region IS NOT NULL)
    """)
    print(f"  silver_dim_territory                          {spark.table('silver_dim_territory').count():>8,} rows")

    spark.sql("""
        CREATE OR REPLACE TABLE silver_dim_date AS
        WITH dates AS (
            SELECT DISTINCT 
                COALESCE(TRY_TO_DATE(Date, 'yyyy-MM-dd'), TRY_TO_DATE(Date, 'MM/dd/yyyy'), TRY_TO_DATE(Date, 'dd-MM-yyyy'), TRY_TO_DATE(Date, 'yyyyMMdd')) AS Date,
                Fiscal_Year, Fiscal_Quarter, Month
            FROM bronze_date WHERE Date IS NOT NULL
        )
        SELECT 
            TRY_CAST(DATE_FORMAT(Date, 'yyyyMMdd') AS INT) AS DateKey, Date,
            CONCAT('FY', REGEXP_EXTRACT(Fiscal_Year, '(\\d{4})', 1)) AS FiscalYear,
            COALESCE(Fiscal_Quarter, CONCAT('FY', REGEXP_EXTRACT(Fiscal_Year, '(\\d{4})', 1), ' Q', CAST(CEIL((MONTH(Date) + CASE WHEN MONTH(Date) >= 7 THEN -6 ELSE 6 END) / 3.0) AS INT))) AS FiscalQuarter,
            Month, TRY_CAST(DATE_FORMAT(Date, 'yyyyMM') AS INT) AS MonthKey
        FROM dates WHERE Date IS NOT NULL
    """)
    print(f"  silver_dim_date                               {spark.table('silver_dim_date').count():>8,} rows")

    spark.sql("""
        CREATE OR REPLACE TABLE silver_dim_salesorder AS
        SELECT 
            TRY_CAST(REGEXP_EXTRACT(SalesOrder, 'SO(\\d+)', 1) AS BIGINT) * 100 + TRY_CAST(REGEXP_EXTRACT(SalesOrderLine, '-\\s*(\\d+)$', 1) AS BIGINT) AS SalesOrderLineKey,
            SalesOrder, SalesOrderLine, Channel
        FROM (SELECT DISTINCT TRIM(Sales_Order) AS SalesOrder, TRIM(Sales_Order_Line) AS SalesOrderLine, COALESCE(INITCAP(TRIM(Channel)), 'Unknown') AS Channel FROM bronze_sales_order WHERE Sales_Order_Line IS NOT NULL)
    """)
    print(f"  silver_dim_salesorder                         {spark.table('silver_dim_salesorder').count():>8,} rows")

    spark.sql("""
        CREATE OR REPLACE TABLE silver_fact_sales AS
        WITH dedup_sales AS (
            SELECT *, ROW_NUMBER() OVER(PARTITION BY Sales_Order_Line ORDER BY Order_Date DESC) AS rn FROM bronze_sales
        )
        SELECT 
            so.SalesOrderLineKey, c.CustomerKey, p.ProductKey, r.ResellerKey, t.SalesTerritoryKey,
            TRY_CAST(DATE_FORMAT(COALESCE(TRY_TO_DATE(s.Order_Date, 'yyyy-MM-dd'), TRY_TO_DATE(s.Order_Date, 'MM/dd/yyyy')), 'yyyyMMdd') AS INT) AS OrderDateKey,
            INITCAP(TRIM(s.Channel)) AS Channel,
            TRY_CAST(ROUND(TRY_CAST(s.Order_Quantity AS DOUBLE), 0) AS INT) AS OrderQuantity,
            TRY_CAST(s.Unit_Price AS DOUBLE) AS UnitPrice,
            CASE WHEN s.Unit_Price_Discount_Pct LIKE '%\\%%' THEN TRY_CAST(REGEXP_EXTRACT(s.Unit_Price_Discount_Pct, '([\\d\\.]+)%', 1) AS DOUBLE) / 100.0 ELSE TRY_CAST(s.Unit_Price_Discount_Pct AS DOUBLE) END AS UnitPriceDiscountPct,
            TRY_CAST(s.Product_Standard_Cost AS DOUBLE) AS ProductStandardCost,
            TRY_CAST(s.Total_Product_Cost AS DOUBLE) AS TotalProductCost,
            TRY_CAST(s.Extended_Amount AS DOUBLE) AS ExtendedAmount,
            TRY_CAST(s.Sales_Amount AS DOUBLE) AS SalesAmount,
            COALESCE(TRY_TO_DATE(s.Order_Date, 'yyyy-MM-dd'), TRY_TO_DATE(s.Order_Date, 'MM/dd/yyyy')) AS OrderDate,
            COALESCE(TRY_TO_DATE(s.Due_Date, 'yyyy-MM-dd'), TRY_TO_DATE(s.Due_Date, 'MM/dd/yyyy')) AS DueDate,
            COALESCE(TRY_TO_DATE(s.Ship_Date, 'yyyy-MM-dd'), TRY_TO_DATE(s.Ship_Date, 'MM/dd/yyyy')) AS ShipDate,
            CASE WHEN TRY_CAST(s.Sales_Amount AS DOUBLE) < 0 THEN 1 ELSE 0 END AS IsReturn
        FROM dedup_sales s
        LEFT JOIN silver_dim_customer c ON TRIM(s.Customer_ID) = c.CustomerNaturalKey
        LEFT JOIN silver_dim_product p ON TRIM(s.SKU) = p.SKU
        LEFT JOIN silver_dim_reseller r ON TRIM(s.Reseller_ID) = r.ResellerNaturalKey
        LEFT JOIN silver_dim_territory t ON INITCAP(TRIM(s.Region)) = t.Region AND INITCAP(TRIM(s.Country)) = t.Country
        LEFT JOIN silver_dim_salesorder so ON TRIM(s.Sales_Order_Line) = so.SalesOrderLine
        WHERE s.rn = 1
    """)
    n_fact = spark.table('silver_fact_sales').count()
    print(f"  silver_fact_sales                             {n_fact:>8,} rows")

    max_date = spark.sql("SELECT MAX(OrderDate) FROM silver_fact_sales").collect()[0][0]
    if max_date: set_watermark("silver_fact_sales", str(max_date), n_fact)

    elapsed = (datetime.now() - start).seconds
    log("silver", rows_out=n_fact)
    print(f"  ✔ Silver done in {elapsed}s")

# COMMAND ----------
# MAGIC %md ## 4. Gold stage

# COMMAND ----------

def run_gold():
    print("\n▓▓▓  STAGE 3/3 — GOLD  ▓▓▓")
    start = datetime.now()

    spark.sql("""
        CREATE OR REPLACE TABLE gold_sales_by_month AS
        WITH base AS (
            SELECT d.FiscalYear, d.FiscalQuarter, d.MonthKey, t.Region, t.Country, t.SalesGroup, p.Category, f.Channel,
                   COUNT(f.SalesOrderLineKey) AS OrderLines, SUM(f.OrderQuantity) AS TotalUnits,
                   ROUND(SUM(f.SalesAmount), 2) AS TotalRevenue, ROUND(AVG(f.SalesAmount), 2) AS AvgOrderValue,
                   ROUND(SUM(f.TotalProductCost), 2) AS TotalCost, COUNT(DISTINCT f.CustomerKey) AS UniqueCustomers
            FROM silver_fact_sales f
            JOIN silver_dim_date d ON f.OrderDateKey = d.DateKey
            LEFT JOIN silver_dim_territory t ON f.SalesTerritoryKey = t.SalesTerritoryKey
            LEFT JOIN silver_dim_product p ON f.ProductKey = p.ProductKey
            WHERE f.IsReturn = 0
            GROUP BY 1, 2, 3, 4, 5, 6, 7, 8
        )
        SELECT *, ROUND(TotalRevenue - TotalCost, 2) AS GrossProfit,
               CASE WHEN TotalRevenue != 0 THEN ROUND(((TotalRevenue - TotalCost) / TotalRevenue) * 100, 1) END AS GrossMarginPct,
               CURRENT_TIMESTAMP() AS _gold_timestamp
        FROM base
    """)
    print(f"  gold_sales_by_month                           {spark.table('gold_sales_by_month').count():>8,} rows")

    spark.sql("""
        CREATE OR REPLACE TABLE gold_product_ranking AS
        WITH tr AS (SELECT SUM(SalesAmount) AS GrandTotal FROM silver_fact_sales WHERE IsReturn = 0),
        agg AS (
            SELECT p.ProductKey, p.SKU, p.Product, p.Model, p.Category, p.Subcategory, p.Color,
                   SUM(f.OrderQuantity) AS UnitsSold, ROUND(SUM(f.SalesAmount), 2) AS TotalRevenue, ROUND(SUM(f.TotalProductCost), 2) AS TotalCost,
                   ROUND(AVG(f.UnitPrice), 2) AS AvgSellingPrice, COUNT(f.SalesOrderLineKey) AS OrderLines
            FROM silver_fact_sales f LEFT JOIN silver_dim_product p ON f.ProductKey = p.ProductKey
            WHERE f.IsReturn = 0 GROUP BY 1, 2, 3, 4, 5, 6, 7
        )
        SELECT a.*, ROUND(a.TotalRevenue - a.TotalCost, 2) AS GrossProfit,
               CASE WHEN a.TotalRevenue != 0 THEN ROUND(((a.TotalRevenue - a.TotalCost) / a.TotalRevenue) * 100, 1) END AS GrossMarginPct,
               CASE WHEN tr.GrandTotal != 0 THEN ROUND((a.TotalRevenue / tr.GrandTotal) * 100, 2) END AS RevSharePct,
               RANK() OVER(ORDER BY a.TotalRevenue DESC) AS RevenueRank,
               CURRENT_TIMESTAMP() AS _gold_timestamp
        FROM agg a CROSS JOIN tr
    """)
    print(f"  gold_product_ranking                          {spark.table('gold_product_ranking').count():>8,} rows")

    spark.sql("""
        CREATE OR REPLACE TABLE gold_customer_summary AS
        WITH rfm AS (
            SELECT CustomerKey, DATEDIFF(CURRENT_DATE(), MAX(OrderDate)) AS RecencyDays, COUNT(SalesOrderLineKey) AS Frequency,
                   ROUND(SUM(SalesAmount), 2) AS LifetimeValue, ROUND(AVG(SalesAmount), 2) AS AvgOrderValue, MAX(OrderDate) AS LastOrderDate
            FROM silver_fact_sales WHERE IsReturn = 0 AND CustomerKey IS NOT NULL GROUP BY CustomerKey
        ),
        scores AS (
            SELECT *,
                CASE WHEN RecencyDays <= 30 THEN 5 WHEN RecencyDays <= 90 THEN 4 WHEN RecencyDays <= 180 THEN 3 WHEN RecencyDays <= 365 THEN 2 ELSE 1 END AS R,
                CASE WHEN Frequency >= 20 THEN 5 WHEN Frequency >= 10 THEN 4 WHEN Frequency >= 5 THEN 3 WHEN Frequency >= 2 THEN 2 ELSE 1 END AS F,
                CASE WHEN LifetimeValue >= 20000 THEN 5 WHEN LifetimeValue >= 10000 THEN 4 WHEN LifetimeValue >= 3000 THEN 3 WHEN LifetimeValue >= 500 THEN 2 ELSE 1 END AS M
            FROM rfm
        )
        SELECT c.CustomerKey, c.CustomerNaturalKey, c.CustomerName, c.City, c.StateProvince, c.CountryRegion,
               COALESCE(s.RecencyDays, 9999) AS RecencyDays, COALESCE(s.Frequency, 0) AS Frequency, COALESCE(s.LifetimeValue, 0.0) AS LifetimeValue, s.AvgOrderValue, s.LastOrderDate,
               CASE WHEN (R+F+M) >= 13 THEN 'Champions' WHEN (R+F+M) >= 10 THEN 'Loyal' WHEN (R+F+M) >= 7 THEN 'Potential' WHEN (R+F+M) >= 5 THEN 'At Risk' ELSE 'Lost' END AS CustomerSegment,
               CURRENT_TIMESTAMP() AS _gold_timestamp
        FROM silver_dim_customer c LEFT JOIN scores s ON c.CustomerKey = s.CustomerKey
    """)
    print(f"  gold_customer_summary                         {spark.table('gold_customer_summary').count():>8,} rows")

    spark.sql("""
        CREATE OR REPLACE TABLE gold_channel_compare AS
        SELECT d.FiscalYear, f.Channel, p.Category, COUNT(f.SalesOrderLineKey) AS OrderLines, SUM(f.OrderQuantity) AS UnitsSold,
               ROUND(SUM(f.SalesAmount), 2) AS TotalRevenue, ROUND(AVG(f.SalesAmount), 2) AS AvgOrderValue, ROUND(SUM(f.TotalProductCost), 2) AS TotalCost, COUNT(DISTINCT f.CustomerKey) AS UniqueCustomers,
               ROUND(SUM(f.SalesAmount) - SUM(f.TotalProductCost), 2) AS GrossProfit,
               CASE WHEN SUM(f.SalesAmount) != 0 THEN ROUND((SUM(f.SalesAmount) - SUM(f.TotalProductCost)) / SUM(f.SalesAmount) * 100, 1) END AS GrossMarginPct,
               CURRENT_TIMESTAMP() AS _gold_timestamp
        FROM silver_fact_sales f JOIN silver_dim_date d ON f.OrderDateKey = d.DateKey LEFT JOIN silver_dim_product p ON f.ProductKey = p.ProductKey
        WHERE f.IsReturn = 0 GROUP BY 1, 2, 3
    """)
    print(f"  gold_channel_compare                          {spark.table('gold_channel_compare').count():>8,} rows")

    elapsed = (datetime.now() - start).seconds
    log("gold")
    print(f"  ✔ Gold done in {elapsed}s")

# COMMAND ----------
# MAGIC %md ## 5. Run the pipeline

# COMMAND ----------

t0 = datetime.now()

print(f"""
╔══════════════════════════════════════════════╗
║  Adventure Works ETL Pipeline                ║
║  Run ID : {RUN_ID:<30}║
║  Mode   : {MODE:<30}║
║  Started: {t0.strftime('%Y-%m-%d %H:%M:%S'):<30}║
╚══════════════════════════════════════════════╝
""")

try:
    run_bronze()
except Exception as e:
    log("bronze", status="FAILED", error=str(e))
    raise

try:
    run_silver()
except Exception as e:
    log("silver", status="FAILED", error=str(e))
    raise

try:
    run_gold()
except Exception as e:
    log("gold", status="FAILED", error=str(e))
    raise

elapsed = (datetime.now() - t0).seconds

print(f"""
╔══════════════════════════════════════════════╗
║  ✅ Pipeline complete ({elapsed}s total)      ║
╚══════════════════════════════════════════════╝
""")

# COMMAND ----------
# MAGIC %md ## 6. Pipeline history

# COMMAND ----------

# MAGIC %sql
# MAGIC SELECT run_id, run_mode, stage, rows_out, status, CAST(started_at AS STRING) AS started_at 
# MAGIC FROM pipeline_run_log ORDER BY started_at DESC LIMIT 20

# COMMAND ----------

# MAGIC %sql
# MAGIC SELECT table_name, CAST(last_order_date AS STRING) AS last_order_date, rows_processed, run_status, CAST(updated_at AS STRING) AS updated_at 
# MAGIC FROM pipeline_watermarks ORDER BY updated_at DESC